In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/size_only/validation/ibm'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

results_directory = GENERATED / 'soay/size_only/validation/ibm'
minimum_replicates = 3

with np.load(results_directory / "comparison_summary.npz") as results:
    years = results["years"]
    statistic_names = results["summary_names"].astype(str)
    exact_mean = results["exact_gp_mean"]
    deeprv_mean = results["deeprv_mean"]
    exact_sd = results["exact_gp_standard_deviation"]
    deeprv_sd = results["deeprv_standard_deviation"]
    exact_count = results["exact_gp_finite_count"]
    deeprv_count = results["deeprv_finite_count"]
    difference_se = results["difference_standard_error"]
    exact_population = results["exact_gp_population_mean"]
    deeprv_population = results["deeprv_population_mean"]
    status_names = results["status_names"].astype(str)
    exact_status = results["exact_gp_status_rate"]
    deeprv_status = results["deeprv_status_rate"]

rows = []

for year_index, year in enumerate(years):
    for statistic_index, statistic in enumerate(statistic_names):
        exact = exact_mean[:, year_index, statistic_index]
        deeprv = deeprv_mean[:, year_index, statistic_index]
        standard_error = difference_se[:, year_index, statistic_index]

        valid = (
            np.isfinite(exact)
            & np.isfinite(deeprv)
            & (exact_count[:, year_index, statistic_index] >= minimum_replicates)
            & (deeprv_count[:, year_index, statistic_index] >= minimum_replicates)
        )

        exact = exact[valid]
        deeprv = deeprv[valid]
        standard_error = standard_error[valid]
        difference = deeprv - exact
        absolute_difference = np.abs(difference)
        exact_iqr = np.percentile(exact, 75) - np.percentile(exact, 25)

        sd_valid = valid & np.isfinite(exact_sd[:, year_index, statistic_index]) & np.isfinite(deeprv_sd[:, year_index, statistic_index]) & (exact_sd[:, year_index, statistic_index] > 0)
        sd_ratio = deeprv_sd[sd_valid, year_index, statistic_index] / exact_sd[sd_valid, year_index, statistic_index]

        se_valid = np.isfinite(standard_error) & (standard_error > 0)

        rows.append({
            "Year": int(year),
            "Statistic": statistic,
            "Parameters Compared": int(valid.sum()),
            "Spearman Correlation": spearmanr(exact, deeprv).statistic,
            "Median Absolute Difference": np.median(absolute_difference),
            "Normalised Median Absolute Difference": np.median(absolute_difference) / exact_iqr if exact_iqr > 0 else np.nan,
            "Median SD Ratio": np.median(sd_ratio) if len(sd_ratio) else np.nan,
            "Within Monte Carlo 95% Interval": np.mean(absolute_difference[se_valid] <= 1.96 * standard_error[se_valid]) if se_valid.any() else np.nan,
        })

accuracy_summary = pd.DataFrame(rows)

selected_year = 1
print(
    accuracy_summary.loc[accuracy_summary["Year"] == selected_year]
    .sort_values("Normalised Median Absolute Difference")
    .to_string(index=False)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

results_directory = GENERATED / 'soay/size_only/validation/ibm'

minimum_replicates = 3

with np.load(results_directory / "comparison_summary.npz") as results:
    years = results["years"]
    statistic_names = results["summary_names"].astype(str)

    exact_mean = results["exact_gp_mean"]
    deeprv_mean = results["deeprv_mean"]

    exact_sd = results["exact_gp_standard_deviation"]

    exact_count = results["exact_gp_finite_count"]
    deeprv_count = results["deeprv_finite_count"]

rows = []

for year_index, year in enumerate(years):
    for statistic_index, statistic in enumerate(statistic_names):

        exact = exact_mean[:, year_index, statistic_index]
        deeprv = deeprv_mean[:, year_index, statistic_index]
        exact_standard_deviation = exact_sd[:, year_index, statistic_index]
        
        valid = (
            np.isfinite(exact)
            & np.isfinite(deeprv)
            & np.isfinite(exact_standard_deviation)
            & (exact_standard_deviation > 0)
        )

        exact_valid = exact[valid]
        deeprv_valid = deeprv[valid]
        exact_sd_valid = exact_standard_deviation[valid]

        absolute_difference = np.abs(deeprv_valid - exact_valid)

        # DeepRV error measured in exact-GP standard deviations
        standardised_absolute_difference = (
            absolute_difference / exact_sd_valid
        )

        rows.append({
            "Year": int(year),
            "Statistic": statistic,
            "Median Absolute Difference": np.median(absolute_difference),
            "Median Standardised Absolute Difference":
                np.median(standardised_absolute_difference),
        })
standardised_accuracy = pd.DataFrame(rows)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

for year in years:
    print(f"\n{'=' * 80}")
    print(f"YEAR {int(year)}")
    print(f"{'=' * 80}")

    print(
        standardised_accuracy.loc[
            standardised_accuracy["Year"] == year
        ]
        .sort_values("Median Standardised Absolute Difference")
        .to_string(index=False)
    )

In [ ]:
mean_accuracy_over_years = (
    standardised_accuracy
    .groupby("Statistic", as_index=False)[
        [
            "Median Absolute Difference",
            "Median Standardised Absolute Difference",
        ]
    ]
    .mean()
    .rename(columns={
        "Median Absolute Difference": "Mean Median Absolute Difference Across Years",
        "Median Standardised Absolute Difference": "Mean Median Standardised Absolute Difference Across Years",
    })
    .sort_values("Mean Median Standardised Absolute Difference Across Years")
)

print(mean_accuracy_over_years.to_string(index=False))

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

results_directory = GENERATED / 'soay/size_only/validation/ibm'

minimum_replicates = 3
selected_statistics = {
    "kl_next_size_distribution":
        "KL Divergence - sizeNext Distribution",
    "chi2_total_survivors_by_size":
        r"$\chi^2$ Distance - Survivors By Size",
    "chi2_total_breeders_by_size":
        r"$\chi^2$ Distance - Breeders By Size",
    "chi2_total_individuals_next_size":
        r"$\chi^2$ Distance - sizeNext Population",
}

with np.load(results_directory / "comparison_summary.npz") as results:
    statistic_names = results["summary_names"].astype(str)
    exact_mean = results["exact_gp_mean"]
    deeprv_mean = results["deeprv_mean"]
    exact_sd = results["exact_gp_standard_deviation"]
    exact_count = results["exact_gp_finite_count"]
    deeprv_count = results["deeprv_finite_count"]

plt.style.use("seaborn-v0_8-white")

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "Arial",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

standardised_differences = {}

for statistic in selected_statistics:
    statistic_index = np.where(statistic_names == statistic)[0][0]

    exact = exact_mean[:, :, statistic_index]
    deeprv = deeprv_mean[:, :, statistic_index]
    gp_sd = exact_sd[:, :, statistic_index]

    valid = (
        np.isfinite(exact)
        & np.isfinite(deeprv)
        & np.isfinite(gp_sd)
        & (gp_sd > 0)
        & (exact_count[:, :, statistic_index] >= minimum_replicates)
        & (deeprv_count[:, :, statistic_index] >= minimum_replicates)
    )

    standardised_differences[statistic] = (
        np.abs(deeprv[valid] - exact[valid]) / gp_sd[valid]
    )

upper_limit = max(
    np.percentile(values, 99.5)
    for values in standardised_differences.values()
)

bin_edges = np.linspace(0, upper_limit, 61)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(5.906, 4.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for axis, (statistic, title) in zip(axes.flat, selected_statistics.items()):
    values = standardised_differences[statistic]
    median_value = np.median(values)

    axis.hist(
        values,
        bins=bin_edges,
        density=True,
        color="#0072B2",
        alpha=0.85,
        edgecolor="none",
    )

    axis.set_title(
    title,
    pad=5,
    )
    
    axis.axvline(
        median_value,
        color="#D55E00",
        linewidth=1.5,
        linestyle="--",
    )

    axis.set_xlim(0, upper_limit)
    axis.tick_params(direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

fig.supxlabel("Standardised Absolute Difference", fontsize=11)
fig.supylabel("Probability Density", fontsize=11)

fig.savefig("selected_statistics_standardised_difference_all_years.pdf", bbox_inches="tight",)
fig.savefig("selected_statistics_standardised_difference_all_years.png",dpi=300, bbox_inches="tight",)

plt.show()

In [ ]:
from pathlib import Path
import textwrap
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr, pearsonr

results_directory = GENERATED / 'soay/size_only/validation/ibm'

year = 1
minimum_replicates = 3

with np.load(results_directory / "comparison_summary.npz") as results:
    years = results["years"]
    statistic_names = results["summary_names"].astype(str)

    exact_mean = results["exact_gp_mean"]
    deeprv_mean = results["deeprv_mean"]

    exact_count = results["exact_gp_finite_count"]
    deeprv_count = results["deeprv_finite_count"]


def readable_name(name):
    name = name.replace("_", " ").title()
    name = (
        name.replace("Ssd", "SSD")
        .replace("Emd", "EMD")
        .replace("Kl", "KL")
        .replace("Chi2", "Chi-Squared")
    )
    return "\n".join(textwrap.wrap(name, 28))


year_index = np.where(years == year)[0][0]

plt.style.use("seaborn-v0_8-white")

fig, axes = plt.subplots(
    4,
    4,
    figsize=(14.4, 12),
    constrained_layout=True
)

for statistic_index, axis in enumerate(axes.flat):

    exact = exact_mean[:, year_index, statistic_index]
    deeprv = deeprv_mean[:, year_index, statistic_index]

    valid = (
        np.isfinite(exact)
        & np.isfinite(deeprv)
        & (
            exact_count[:, year_index, statistic_index]
            >= minimum_replicates
        )
        & (
            deeprv_count[:, year_index, statistic_index]
            >= minimum_replicates
        )
    )

    exact = exact[valid]
    deeprv = deeprv[valid]

    # Correlations
    pearson_correlation = pearsonr(exact, deeprv).statistic
    spearman_correlation = spearmanr(exact, deeprv).statistic

    # Use the 99.5th percentile so extreme values do not dominate the axes
    upper_limit = np.percentile(
        np.concatenate([exact, deeprv]),
        99.5
    )
    upper_limit = max(upper_limit, 1e-12)

    # Parameter-level Exact GP vs DeepRV mean distances
    axis.scatter(
        exact,
        deeprv,
        s=3,
        alpha=0.08,
        color="#0072B2",
        rasterized=True
    )

    # Identity line: Exact GP = DeepRV
    axis.plot(
        [0, upper_limit],
        [0, upper_limit],
        color="#D55E00",
        linewidth=1.4
    )

    axis.set_xlim(0, upper_limit)
    axis.set_ylim(0, upper_limit)

    axis.set_title(
        f"{readable_name(statistic_names[statistic_index])}\n"
        f"Pearson $r$ = {pearson_correlation:.3f}, "
        f"Spearman $\\rho$ = {spearman_correlation:.3f}",
        fontsize=9
    )

    axis.tick_params(
        labelsize=8,
        direction="out"
    )

    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)


fig.supxlabel(
    "Exact GP Mean Distance",
    fontsize=14
)

fig.supylabel(
    "DeepRV Mean Distance",
    fontsize=14
)

fig.suptitle(
    f"Parameter-Level Distance Agreement: Year {year}",
    fontsize=16
)

fig.savefig(f"1-dim_parameter_distance_scatter_year_{year}.pdf", bbox_inches="tight")
fig.savefig(f"1-dim_parameter_distance_scatter_year_{year}.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
from pathlib import Path
import textwrap
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

results_directory = GENERATED / 'soay/size_only/validation/ibm'
year = 1
minimum_replicates = 3
lengthscale_name = "surv_ls"

with np.load(results_directory / "comparison_summary.npz") as results:
    years = results["years"]
    statistic_names = results["summary_names"].astype(str)
    parameter_names = results["parameter_names"].astype(str)
    parameter_values = results["parameter_values"]
    exact_mean = results["exact_gp_mean"]
    deeprv_mean = results["deeprv_mean"]
    exact_count = results["exact_gp_finite_count"]
    deeprv_count = results["deeprv_finite_count"]

def readable_name(name):
    name = name.replace("_", " ").title()
    name = name.replace("Ssd", "SSD").replace("Emd", "EMD").replace("Kl", "KL").replace("Chi2", "Chi-Squared")
    return "\n".join(textwrap.wrap(name, 28))

lengthscale_labels = {
    "growth_ls": "Growth Lengthscale",
    "surv_ls": "Survival Lengthscale",
    "repr_ls": "Reproduction Lengthscale",
    "rcsz_ls": "Recruit Size Lengthscale",
}

year_index = np.where(years == year)[0][0]
lengthscale_index = np.where(parameter_names == lengthscale_name)[0][0]
lengthscales = parameter_values[:, lengthscale_index]
quartile_edges = np.quantile(lengthscales, [0, 0.25, 0.5, 0.75, 1])

upper_limits = []
for statistic_index in range(len(statistic_names)):
    exact = exact_mean[:, year_index, statistic_index]
    deeprv = deeprv_mean[:, year_index, statistic_index]
    valid = np.isfinite(exact) & np.isfinite(deeprv) & (exact_count[:, year_index, statistic_index] >= minimum_replicates) & (deeprv_count[:, year_index, statistic_index] >= minimum_replicates)
    upper_limits.append(max(np.percentile(np.concatenate([exact[valid], deeprv[valid]]), 99.5), 1e-12))

plt.style.use("seaborn-v0_8-white")

for quartile in range(4):
    lower = quartile_edges[quartile]
    upper = quartile_edges[quartile + 1]
    range_mask = (lengthscales >= lower) & (lengthscales <= upper) if quartile == 3 else (lengthscales >= lower) & (lengthscales < upper)

    fig, axes = plt.subplots(4, 4, figsize=(14.4, 12), constrained_layout=True)

    for statistic_index, axis in enumerate(axes.flat):
        exact = exact_mean[:, year_index, statistic_index]
        deeprv = deeprv_mean[:, year_index, statistic_index]
        valid = range_mask & np.isfinite(exact) & np.isfinite(deeprv) & (exact_count[:, year_index, statistic_index] >= minimum_replicates) & (deeprv_count[:, year_index, statistic_index] >= minimum_replicates)

        exact = exact[valid]
        deeprv = deeprv[valid]
        correlation = spearmanr(exact, deeprv).statistic
        upper_limit = upper_limits[statistic_index]

        axis.scatter(exact, deeprv, s=4, alpha=0.12, color="#0072B2", rasterized=True)
        axis.plot([0, upper_limit], [0, upper_limit], color="#D55E00", linewidth=1.4)
        axis.set_xlim(0, upper_limit)
        axis.set_ylim(0, upper_limit)
        axis.set_title(f"{readable_name(statistic_names[statistic_index])}\nSpearman $\\rho$ = {correlation:.3f}", fontsize=9)
        axis.tick_params(labelsize=8, direction="out")
        axis.grid(False)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    fig.supxlabel("Exact GP Mean Distance", fontsize=14)
    fig.supylabel("DeepRV Mean Distance", fontsize=14)
    fig.suptitle(f"Parameter-Level Distance Agreement: Year {year}\n{lengthscale_labels[lengthscale_name]}, Quartile {quartile + 1}: {lower:.3f}–{upper:.3f}", fontsize=16)

    output_name = f"year_{year}_{lengthscale_name}_quartile_{quartile + 1}"
    plt.show()